# Lab 6: Deep learning for NLP

Guidelines: https://github.com/MohammedMosuily/NLP_Lab6

Run cells from top to bottom. Dataset paths are relative to this folder.

## Load, preprocess, and split Yelp reviews

In [1]:
import re
import numpy as np
import pandas as pd
import torch
from torch import nn
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

torch.manual_seed(42)
df = pd.read_csv("dataset/07-yelp-dataset.txt", sep="\t", names=["review", "label"])

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return " ".join(text.split())

df["review"] = df["review"].apply(preprocess)
X_train, X_test, y_train, y_test = train_test_split(
    df["review"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)
print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 800
Testing reviews: 200


## Task 1: Word2Vec review vectors
Train embeddings on training reviews only. Average known word vectors for each review. An empty review or one with only unseen words receives a zero vector.

In [2]:
train_tokens = [text.split() for text in X_train]
test_tokens = [text.split() for text in X_test]
w2v = Word2Vec(train_tokens, vector_size=200, window=5, min_count=1, sg=1, workers=1, seed=42)

def review_vector(tokens):
    vectors = [w2v.wv[word] for word in tokens if word in w2v.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(200, dtype=np.float32)

X_train_tensor = torch.tensor(np.array([review_vector(t) for t in train_tokens]), dtype=torch.float32)
X_test_tensor = torch.tensor(np.array([review_vector(t) for t in test_tokens]), dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).reshape(-1, 1)
print(X_train_tensor.shape, X_test_tensor.shape)

torch.Size([800, 200]) torch.Size([200, 200])


## Task 2: Three-hidden-layer neural network
Architecture: 200 → 128 → 64 → 32 → 1. Use ReLU, BCEWithLogitsLoss, Adam at 0.001, and 15 epochs. Clear gradients before each update.

In [3]:
model = nn.Sequential(
    nn.Linear(200, 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 1),
)
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
for epoch in range(15):
    model.train()
    optimizer.zero_grad()
    loss = loss_function(model(X_train_tensor), y_train_tensor)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch + 1}/15 - Loss: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    predictions = (torch.sigmoid(model(X_test_tensor)) >= 0.5).int().numpy().flatten()
print("Test accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions, target_names=["Negative", "Positive"], zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, predictions))

Epoch 1/15 - Loss: 0.6958
Epoch 2/15 - Loss: 0.6953
Epoch 3/15 - Loss: 0.6949
Epoch 4/15 - Loss: 0.6946
Epoch 5/15 - Loss: 0.6943
Epoch 6/15 - Loss: 0.6941
Epoch 7/15 - Loss: 0.6938
Epoch 8/15 - Loss: 0.6937
Epoch 9/15 - Loss: 0.6935
Epoch 10/15 - Loss: 0.6934
Epoch 11/15 - Loss: 0.6933
Epoch 12/15 - Loss: 0.6933
Epoch 13/15 - Loss: 0.6932
Epoch 14/15 - Loss: 0.6932
Epoch 15/15 - Loss: 0.6932
Test accuracy: 0.49
              precision    recall  f1-score   support

    Negative       0.49      0.98      0.66       100
    Positive       0.00      0.00      0.00       100

    accuracy                           0.49       200
   macro avg       0.25      0.49      0.33       200
weighted avg       0.25      0.49      0.33       200

Confusion matrix:
[[ 98   2]
 [100   0]]


The measured accuracy is close to chance. This completes the requested experiment, but the result does not demonstrate a reliable sentiment model. Possible follow-up experiments include pretrained embeddings or more training epochs.